# OliVe / AB-Float：非均匀位宽映射

**对应博客篇目**：PTQ 系列第 12 篇《OliVe：硬件友好的离群值-受害者配对量化——当算法思路被焊进数据格式》

---

## 原理概述

OliVe 的核心思想是 **离群值-受害者配对（OVP）**：将激活按固定步长两两配对，
若一对中出现离群值，则邻居「牺牲」（victim，置零腾出空间），离群值独占槽位并用 **abfloat**
宽格式编码；若一对都是正常值，则各拿一份窄格式（如 E2M1 四级尾数）。

**abfloat** 是迷你浮点的变体——通过指数偏移让全部编码值跳过正常数值区间：
可表示幅值 $= \{1,\,1.25,\,1.5,\,1.75\} \times 2^z$，$z \in [z_{\min}, z_{\max}]$，
使得 $2^{z_{\min}}$ 高于所有正常值——低于此阈值的一律冲刷为零。

本 demo 演示三个核心点：
1. **给定码字集合的非均匀最近邻映射**——abfloat 用对数间隔的稀疏码字逼近大值
2. **贪心位分配**——敏感通道（离群值）优先获得宽格式码字，正常通道共享窄网格
3. **同 bit 预算下均匀 vs 非均匀 SQNR**——三种方案的重构误差对比

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
rng = np.random.default_rng(SEED)
print(f"随机种子: SEED={SEED}")

随机种子: SEED=0


## 第一步：构造 toy 数据

模拟 LLM 激活的典型分布：绝大多数元素来自 $\mathcal{N}(0,1)$（正常值），
随机挑出约 2% 的元素赋予 16--64 倍均值作为离群值。
这种「少数通道幅值远大于其余」的形态是 Transformer 激活的标志性特征。

In [2]:
# 构造激活向量：正常值 N(0,1) + 2% 离群值 U(16,64)
n = 4096
x = rng.normal(0, 1.0, n)
pos = rng.choice(n, int(n * 0.02), replace=False)
x[pos] = rng.uniform(16.0, 64.0, len(pos))

# 离群判定阈值（实际中由校准统计确定，此处用真值 8.0）
is_out = np.abs(x) > 8.0

print(f"向量长度: {n}，离群元素: {is_out.sum()} ({is_out.mean()*100:.1f}%)")
print(f"正常值 |x| max: {np.abs(x[~is_out]).max():.4f}")
print(f"离群值范围: [{x[is_out].min():.2f}, {x[is_out].max():.2f}]")

# 正常值自己的共享指数（用于窄格式网格）
k_normal = int(np.floor(np.log2(np.abs(x[~is_out]).max() / 6)))
k_shared = int(np.floor(np.log2(np.abs(x).max() / 6)))
print(f"k_normal={k_normal}（仅正常值）vs 共享 k={k_shared}（被离群值劫持）")

向量长度: 4096，离群元素: 81 (2.0%)
正常值 |x| max: 3.8994
离群值范围: [16.52, 63.88]
k_normal=-1（仅正常值）vs 共享 k=3（被离群值劫持）


## 第二步：量化工具函数

三个核心编码器 + 一个误差度量：

- **`e2m1_shared(x)`**：E2M1 四级尾数 $\{0,\,0.5,\,1,\,1.5,\,2,\,3,\,4,\,6\} \times 2^k$，
  $k$ 由输入 absmax 决定——这是「全员共享窄网格」的基线
- **`narrow_saturate(x, k)`**：同样的窄网格，但 $k$ 锁定为正常值指数，
  超界值被 clip 到网格最大电平——模拟「没有宽格式、直接饱和」
- **`abfloat(x, z_lo, z_hi)`**：abfloat 编码，可表示幅值 $= \{1,1.25,1.5,1.75\} \times 2^z$，
  幅值 $< 2^{z_{\min}}$ 一律冲刷为零——**非均匀码字**的典型代表
- **`rel_err(a, b)`**：Frobenius 范数相对误差

In [3]:
# abfloat 尾数电平表（固定 4 级）
LEVELS_M = np.array([1.0, 1.25, 1.5, 1.75])


def e2m1_shared(x):
    """E2M1 共享网格：k 由全体元素 absmax 决定（被离群值劫持）"""
    levels = np.array([0, .5, 1, 1.5, 2, 3, 4, 6])
    k = int(np.floor(np.log2(np.abs(x).max() / 6)))
    grid = levels * 2.0 ** k
    return grid[np.argmin(np.abs(x[:, None] - grid[None, :]), axis=1)]


def narrow_saturate(x, k_normal):
    """窄格式 + 饱和：k 锁定为正常值指数，超界 clip"""
    levels = np.array([0, .5, 1, 1.5, 2, 3, 4, 6])
    grid = levels * 2.0 ** k_normal
    return np.clip(
        grid[np.argmin(np.abs(x[:, None] - grid[None, :]), axis=1)],
        -grid[-1], grid[-1],
    )


def abfloat(x, z_lo=3, z_hi=10):
    """abfloat 编码：可表示幅值 = {1,1.25,1.5,1.75}×2^z
    关键性质：幅值 < 2^z_lo 一律冲刷为 0（跳过正常区间）"""
    xq = np.zeros_like(x)
    nz = np.abs(x) > 0
    a = np.abs(x[nz])
    z = np.clip(np.floor(np.log2(a)), z_lo, z_hi).astype(int)
    scale = 2.0 ** z
    m = LEVELS_M[np.argmin(np.abs(a / scale - LEVELS_M[:, None]), axis=0)]
    xq[nz] = np.sign(x[nz]) * m * scale
    return xq


def rel_err(a, b):
    """Frobenius 范数相对误差"""
    return float(np.linalg.norm(a - b) / np.linalg.norm(b))


print("工具函数已定义: e2m1_shared, narrow_saturate, abfloat, rel_err")

工具函数已定义: e2m1_shared, narrow_saturate, abfloat, rel_err


## 第三步：编码原理可视化——非均匀码字的最近邻映射

abfloat 的核心是 **非均匀码字集合**：码字在对数尺度上均匀分布，
但在线性尺度上呈现「低端稀疏、高端密集」的特性。
给定一个输入值，量化就是找码字集合中距离最近的那个——即**非均匀最近邻映射**。

下图展示 abfloat 在不同指数下的码字覆盖，以及窄网格在离群值区间的饱和现象。

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 左图：abfloat 码字集合（非均匀最近邻映射）──
ax = axes[0]
# 绘制各指数层的码字位置
for z in range(3, 11):
    codes = LEVELS_M * 2**z
    ax.scatter(codes, [z] * len(codes), c="#4C72B0", s=18, zorder=3)
    ax.scatter(-codes, [z] * len(codes), c="#4C72B0", s=18, zorder=3)

# 标注离群值区间
ax.axvspan(16, 64, alpha=0.12, color="#C44E52", label="离群值区间 [16,64]")
ax.axvspan(-64, -16, alpha=0.12, color="#C44E52")
ax.axvline(8, color="gray", ls="--", lw=1, alpha=0.6, label=f"2^z_min = 8（冲刷阈值）")
ax.axvline(-8, color="gray", ls="--", lw=1, alpha=0.6)
ax.set_xlabel("可表示数值")
ax.set_ylabel("指数 z")
ax.set_title("abfloat 码字集合：对数间隔的非均匀最近邻映射")
ax.set_xlim(-200, 200)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── 右图：窄网格 vs abfloat 在离群值区间的覆盖 ──
ax = axes[1]
# 窄网格（k_normal=-1）的码字
levels_n = np.array([0, .5, 1, 1.5, 2, 3, 4, 6])
narrow_codes = levels_n * 2**k_normal
ax.vlines(narrow_codes, 0, 1, colors="#DD8452", lw=2, label=f"窄网格 k={k_normal}")
ax.vlines(-narrow_codes, 0, 1, colors="#DD8452", lw=2)

# abfloat 码字（z=3~10 全部正侧）
ab_codes_all = np.concatenate([LEVELS_M * 2**z for z in range(3, 11)])
ax.vlines(ab_codes_all, 0, 0.7, colors="#4C72B0", lw=1.5, alpha=0.7,
          label="abfloat 码字 z∈[3,10]")

ax.axvspan(16, 64, alpha=0.12, color="#C44E52", label="离群值区间 [16,64]")
ax.set_xlim(0, 80)
ax.set_ylim(0, 1.15)
ax.set_xlabel("可表示数值")
ax.set_title("窄网格在离群值区间码字稀疏→饱和；abfloat 覆盖密集")
ax.legend(fontsize=8)
ax.set_yticks([])
ax.grid(alpha=0.3, axis="x")

fig.tight_layout()
plt.show()

/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2295529423.py:45: UserWarning: Glyph 21487 (\N{CJK UNIFIED IDEOGRAPH-53EF}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2295529423.py:45: UserWarning: Glyph 34920 (\N{CJK UNIFIED IDEOGRAPH-8868}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2295529423.py:45: UserWarning: Glyph 31034 (\N{CJK UNIFIED IDEOGRAPH-793A}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2295529423.py:45: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2295529423.py:45: UserWarning: Glyph 20540 (\N{CJK UNIFIED IDEOGRAPH-503C}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16w

## 第四步：Demo A——同槽位预算下的三种编码对比

三种方案在**相同的位宽预算**下编码同一向量：

| 方案 | 正常值编码 | 离群值编码 | 策略 |
|:---|:---|:---|:---|
| 方案 1 全员共享窄网格 | E2M1，$k$ 被离群值劫持 | 同上 | 无差别对待 |
| 方案 2 victim+饱和 | E2M1，$k$ 锁定正常值 | 饱和到窄网格上限 | 牺牲了但没给宽格式 |
| 方案 3 OVP | E2M1，干净的 $k$ | abfloat 宽格式 | 牺牲+格式升级 |

固定步长相邻配对（奇偶位），(outlier, normal) 对中 normal 位作为 victim 置零。
这演示了**贪心位分配**：敏感通道优先获得宽格式码字资源。

In [5]:
# ── 方案 1：全员共享窄网格 ──
q1 = e2m1_shared(x)

# ── 方案 2：victim 置零 + 窄格式饱和 ──
q2 = x.copy()
q2[~is_out] = narrow_saturate(x[~is_out], k_normal)
q2[is_out] = narrow_saturate(x[is_out], k_normal)

# ── 方案 3：OVP（细网格 + abfloat + victim）──
q3 = x.copy()
q3[~is_out] = e2m1_shared(x[~is_out])
q3[is_out] = abfloat(x[is_out])

# ── 固定步长相邻配对，标记 victim ──
n_victim = 0
for a_ in range(0, n, 2):
    b_ = a_ + 1
    if is_out[a_] != is_out[b_]:
        q2[b_ if is_out[a_] else a_] = 0.0
        q3[b_ if is_out[a_] else a_] = 0.0
        n_victim += 1

# ── 结果对比 ──
print(f"[DemoA] 同槽位预算下的重构相对误差")
print(f"  离群 {is_out.mean()*100:.0f}%，victim 约 {n_victim/n*100:.1f}%")
print(f"  k_normal={k_normal}（仅正常值）vs 共享 k={k_shared}（被劫持）")
print(f"")
print(f"  方案1 全员共享窄网格            : {rel_err(q1, x):.4f}")
print(f"  方案2 victim置零+窄格式饱和     : {rel_err(q2, x):.4f}")
print(f"  方案3 OVP(细网格+abfloat+victim): {rel_err(q3, x):.4f}")

[DemoA] 同槽位预算下的重构相对误差
  离群 2%，victim 约 1.9%
  k_normal=-1（仅正常值）vs 共享 k=3（被劫持）

  方案1 全员共享窄网格            : 0.2215
  方案2 victim置零+窄格式饱和     : 0.9319
  方案3 OVP(细网格+abfloat+victim): 0.1329


## 第五步：Demo B——abfloat 的范围覆盖与离群值表示精度

abfloat 的可表示范围 $[\,8,\,1792\,]$ 且 $(0,8)$ 区间**字面上没有任何编码值**——
"跳过正常区间"不是比喻，而是该数制下的物理事实。

这展示了非均匀量化的另一面：**用范围裁剪（range pruning）换取大值区间的密集覆盖**。
正常值有自己的细网格服务，宽格式可以放心浪费低端区间。

In [6]:
# ── abfloat 可表示范围 ──
ab_min = LEVELS_M[0] * 2**3
ab_max = LEVELS_M[-1] * 2**10
print(f"[DemoB] 可表示范围对比")
print(f"  abfloat(z∈[3,10]) 覆盖: [{ab_min:.1f}, {ab_max:.0f}]")
print(f"  且 (0, {2**3}) 区间无任何编码 —— '跳过'正常区间")

# ── 离群值逐元素表示误差 ──
v_test = rng.uniform(16.0, 64.0, 200)
errs_ab = [rel_err(abfloat(np.array([v])), np.array([v])) for v in v_test]
errs_sat = [rel_err(narrow_saturate(np.array([v]), k_normal), np.array([v]))
            for v in v_test]

print(f"")
print(f"  离群值(16~64)平均表示误差:")
print(f"    abfloat      = {np.mean(errs_ab)*100:.1f}%")
print(f"    窄格式饱和   = {np.mean(errs_sat)*100:.1f}%")

[DemoB] 可表示范围对比
  abfloat(z∈[3,10]) 覆盖: [8.0, 1792]
  且 (0, 8) 区间无任何编码 —— '跳过'正常区间

  离群值(16~64)平均表示误差:
    abfloat      = 4.8%
    窄格式饱和   = 91.7%


## 第六步：可视化——误差对比与离群值表示精度

左图：三种方案的重构相对误差柱状图。
右图：200 个离群值在 abfloat 与窄格式饱和下的逐元素相对误差分布。

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 左图：三种方案重构误差对比 ──
ax = axes[0]
names = ["方案1\n全员共享窄网格", "方案2\nvictim+饱和", "方案3\nOVP(细网格+abfloat)"]
errs = [rel_err(q1, x), rel_err(q2, x), rel_err(q3, x)]
colors = ["#DD8452", "#C44E52", "#55A868"]
bars = ax.bar(names, errs, color=colors, width=0.55)
for b, v in zip(bars, errs):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01,
            f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("重构相对误差")
ax.set_title(f"同 bit 预算：非均匀映射(OVP) vs 均匀量化")
ax.set_ylim(0, max(errs) * 1.15)
ax.grid(alpha=0.3, axis="y")

# ── 右图：离群值逐元素误差分布 ──
ax = axes[1]
ax.hist(errs_ab, bins=25, alpha=0.7, color="#4C72B0",
        label=f"abfloat (均值 {np.mean(errs_ab)*100:.1f}%)", edgecolor="white")
ax.hist(errs_sat, bins=25, alpha=0.7, color="#C44E52",
        label=f"窄格式饱和 (均值 {np.mean(errs_sat)*100:.1f}%)", edgecolor="white")
ax.set_xlabel("单元素相对误差")
ax.set_ylabel("频次")
ax.set_title("离群值(16~64) 逐元素表示误差")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2819943835.py:29: UserWarning: Glyph 26041 (\N{CJK UNIFIED IDEOGRAPH-65B9}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2819943835.py:29: UserWarning: Glyph 26696 (\N{CJK UNIFIED IDEOGRAPH-6848}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2819943835.py:29: UserWarning: Glyph 20840 (\N{CJK UNIFIED IDEOGRAPH-5168}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2819943835.py:29: UserWarning: Glyph 21592 (\N{CJK UNIFIED IDEOGRAPH-5458}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20352/2819943835.py:29: UserWarning: Glyph 20849 (\N{CJK UNIFIED IDEOGRAPH-5171}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16w

## 结果解读

### 关键数字汇总

| 方案 | 重构误差 | 说明 |
|:---|:---:|:---|
| 方案 1 全员共享窄网格 | **0.2215** | 离群值把共享指数从 $k=-1$ 撑到 $k=3$，正常值只剩粗电平 |
| 方案 2 victim+饱和 | **0.9319** | 离群值承载约 97% 能量，饱和后等于丢弃整个张量 |
| 方案 3 OVP | **0.1329** | 正常对拿回细网格，离群值由 abfloat 服务，比方案 1 降低约 40% |

| 指标 | 数值 |
|:---|:---:|
| abfloat 覆盖范围 | $[8,\,1792]$，$(0,8)$ 无编码 |
| 离群值 abfloat 表示误差 | **4.8%** |
| 离群值窄格式饱和误差 | **91.7%** |

### 核心结论

1. **victim 的意义必须由格式升级来兑现**：方案 2 牺牲了邻居却只给窄格式饱和，
   误差 0.9319 全场最差——「象征性的特殊待遇」不如不做。
2. **非均匀码字天然适配离群分布**：abfloat 用对数间隔的稀疏码字覆盖大值区间，
   精度损失仅 4.8%，远优于均匀网格的饱和截断。
3. **固定配对保证零索引零 gather**：编解码只依赖本对内两个元素，
   没有跨对信息——这是 OliVe 硬件友好的全部含义。

### 方法局限

1. **victim 丢弃是硬损失**：邻居信息永久丢失，离群占比升高时精度对阈值 $\tau$ 敏感。
2. **固定步长配对的机会主义**：(outlier, outlier) 对处理尴尬，需要退化策略兜底。
3. **abfloat 指数范围需校准**：$z \in [3,10]$ 针对特定分布硬编码，
   换一个模型或层可能需要重新选择偏移量。
4. **硬件收益在本 demo 中不可见**：真正的吞吐优势来自 TensorCore 前置的门级解码电路和
   专用 MMA 指令，numpy 模拟仅度量编码格式本身的精度潜力。

### 参考文献

- OliVe: arXiv:2304.07493 (OSDI'23)
- LLM.int8(): arXiv:2208.07339
- ATOM: arXiv:2310.19102